In [4]:
!pip install pandas scipy ortools folium

  Using cached scipy-1.17.1-cp311-cp311-win_amd64.whl.metadata (60 kB)
  Using cached folium-0.20.0-py2.py3-none-any.whl.metadata (4.2 kB)
  Using cached branca-0.8.2-py3-none-any.whl.metadata (1.7 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached requests-2.33.1-py3-none-any.whl.metadata (4.8 kB)
  Using cached xyzservices-2026.3.0-py3-none-any.whl.metadata (4.1 kB)
  Using cached markupsafe-3.0.3-cp311-cp311-win_amd64.whl.metadata (2.8 kB)
  Using cached charset_normalizer-3.4.7-cp311-cp311-win_amd64.whl.metadata (41 kB)
  Using cached idna-3.11-py3-none-any.whl.metadata (8.4 kB)
  Using cached urllib3-2.6.3-py3-none-any.whl.metadata (6.9 kB)
  Using cached certifi-2026.2.25-py3-none-any.whl.metadata (2.5 kB)
Using cached scipy-1.17.1-cp311-cp311-win_amd64.whl (36.6 MB)
Using cached folium-0.20.0-py2.py3-none-any.whl (113 kB)
Using cached branca-0.8.2-py3-none-any.whl (26 kB)
Using cached jinja2-3.1.6-py3-none-any.whl (134 kB)
Using cached markupsafe-3

In [3]:
from ortools.constraint_solver import pywrapcp
print("All working ✅")

All working ✅


In [5]:
import pandas as pd
from scipy.spatial.distance import cdist
import folium

from ortools.constraint_solver import routing_enums_pb2
from ortools.constraint_solver import pywrapcp

In [6]:
data = {
    'location': [
        'Depot (Hitech City)',
        'Gachibowli',
        'Madhapur',
        'Kondapur',
        'Banjara Hills',
        'Jubilee Hills',
        'Charminar',
        'Secunderabad',
        'Begumpet',
        'Ameerpet'
    ],
    
    'lat': [
        17.4435, 17.4401, 17.4483, 17.4698, 17.4126,
        17.4239, 17.3616, 17.4399, 17.4443, 17.4375
    ],
    
    'lon': [
        78.3772, 78.3489, 78.3915, 78.3639, 78.4482,
        78.4738, 78.4747, 78.4983, 78.4625, 78.4483
    ]
}

df = pd.DataFrame(data)
df

,location,lat,lon
0,Depot (Hitech City),17.4435,78.3772
1,Gachibowli,17.4401,78.3489
2,Madhapur,17.4483,78.3915
3,Kondapur,17.4698,78.3639
4,Banjara Hills,17.4126,78.4482
5,Jubilee Hills,17.4239,78.4738
6,Charminar,17.3616,78.4747
7,Secunderabad,17.4399,78.4983
8,Begumpet,17.4443,78.4625
9,Ameerpet,17.4375,78.4483


In [7]:
coords = df[['lat', 'lon']].values
distance_matrix = cdist(coords, coords, metric='euclidean')

distance_matrix

array([[0.        , 0.02850351, 0.0150841 , 0.02947168, 0.07743262,
        0.09856835, 0.12733366, 0.1211535 , 0.08530375, 0.07135272],
       [0.02850351, 0.        , 0.04338202, 0.03327296, 0.10303757,
        0.12594622, 0.14828314, 0.14940013, 0.11367761, 0.099434  ],
       [0.0150841 , 0.04338202, 0.        , 0.03498585, 0.06700284,
        0.08584084, 0.12016293, 0.10712983, 0.07111259, 0.05781764],
       [0.02947168, 0.03327296, 0.03498585, 0.        , 0.10187409,
        0.11910004, 0.1548673 , 0.13768577, 0.10184405, 0.09036952],
       [0.07743262, 0.10303757, 0.06700284, 0.10187409, 0.        ,
        0.02798303, 0.05747391, 0.05705524, 0.03477614, 0.0249002 ],
       [0.09856835, 0.12594622, 0.08584084, 0.11910004, 0.02798303,
        0.        , 0.0623065 , 0.02926175, 0.02332059, 0.0289    ],
       [0.12733366, 0.14828314, 0.12016293, 0.1548673 , 0.05747391,
        0.0623065 , 0.        , 0.08177928, 0.08359504, 0.08036025],
       [0.1211535 , 0.14940013, 0.1071298

In [8]:
def create_data_model():
    return {
        'distance_matrix': distance_matrix,
        'num_vehicles': 2,
        'depot': 0
    }

data_model = create_data_model()

manager = pywrapcp.RoutingIndexManager(
    len(data_model['distance_matrix']),
    data_model['num_vehicles'],
    data_model['depot']
)

routing = pywrapcp.RoutingModel(manager)

In [9]:
def distance_callback(from_index, to_index):
    return int(
        data_model['distance_matrix'][
            manager.IndexToNode(from_index)
        ][
            manager.IndexToNode(to_index)
        ] * 1000
    )

transit_callback_index = routing.RegisterTransitCallback(distance_callback)
routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)

In [10]:
search_parameters = pywrapcp.DefaultRoutingSearchParameters()
search_parameters.first_solution_strategy = (
    routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
)

solution = routing.SolveWithParameters(search_parameters)

In [11]:
routes = []

for vehicle_id in range(data_model['num_vehicles']):
    index = routing.Start(vehicle_id)
    route = []

    while not routing.IsEnd(index):
        route.append(manager.IndexToNode(index))
        index = solution.Value(routing.NextVar(index))

    route.append(manager.IndexToNode(index))
    routes.append(route)

    print(f"\nVehicle {vehicle_id} Route:")
    for i in route:
        print(df.iloc[i]['location'], end=" → ")


Vehicle 0 Route:
Depot (Hitech City) → Depot (Hitech City) → 
Vehicle 1 Route:
Depot (Hitech City) → Gachibowli → Kondapur → Madhapur → Ameerpet → Begumpet → Secunderabad → Jubilee Hills → Charminar → Banjara Hills → Depot (Hitech City) → 

In [12]:
map_ = folium.Map(location=[17.3850, 78.4867], zoom_start=11)

# markers
for i, row in df.iterrows():
    folium.Marker([row['lat'], row['lon']], popup=row['location']).add_to(map_)

# routes
colors = ['blue', 'red']

for idx, route in enumerate(routes):
    points = [(df.iloc[i]['lat'], df.iloc[i]['lon']) for i in route]
    folium.PolyLine(points, color=colors[idx]).add_to(map_)

map_